In [ ]:
# int_cols = ["order_number", "line_item", "customerkey", "storekey", "productkey", "quantity"]
# date_cols = ["order_date", "delivery_date"]
# string_cols = ["currency_code"]
#
#
# def str_trim(df, cols):
#     for col in df.columns:
#         if col in cols:
#             df = df.withColumn(col, F.trim(F.col(col)))
#
#     return df
#
# def to_integer(df, cols):
#
#     for col in df.columns:
#         if col in cols:
#             df = df.withColumn(col, F.col(col).cast("int"))
#
#     return df
#
#
# def to_date(df, cols, format="M/d/yyyy"):
#
#
#     for col in df.columns:
#         if col in cols:
#             before_count = df.filter(F.col(col).isNotNull()).count()
#             df = df.withColumn(col, F.try_to_date(F.col(col), format))
#             after_count = df.filter(F.col(col).isNotNull()).count()
#
#             if after_count < before_count:
#                 raise ValueError(f"{col}: {before_count - after_count} values failed to parse as {format}")
#
#     return df
#
# sales_df_raw = str_trim(sales_df_raw, string_cols)
# sales_df_raw = to_integer(sales_df_raw, int_cols)
# sales_df_raw = to_date(sales_df_raw, date_cols)
#
# sales_df_raw.printSchema()


In [ ]:
from databricks.connect import DatabricksSession
from pyspark.sql import functions as F

In [ ]:
spark = DatabricksSession.builder.serverless().profile("azure").getOrCreate()

In [ ]:
customers_bronze = spark.table("electronics.bronze.customers")
sales_bronze  = spark.table("electronics.bronze.sales")
products_bronze = spark.table("electronics.bronze.products")
stores_bronze = spark.table("electronics.bronze.stores")
exchange_rates_bronze = spark.table("electronics.bronze.exchange_rates")

In [ ]:
customers_bronze.show(5, truncate=False)

In [ ]:
customers_bronze.printSchema()

In [ ]:
SILVER_TYPES = {
    "customers" : {
        "int": ["customerkey"],
        "date": ["birthday"],
        "string": ["gender", "name", "city", "state_code", "state", "zip_code", "country", "continent"]
    },
    "sales": {
        "int": ["order_number", "line_item", "customerkey", "storekey", "productkey", "quantity"],
        "date": ["order_date", "delivery_date"],
        "string": ["currency_code"]
    },
    "products": {
        "int": ["productkey", "subcategorykey", "categorykey"],
        "string": ["product_name", "brand", "color", "subcategory", "category"]
    },
    "stores": {
        "int": ["storekey", "square_meters"],
        "date": ["open_date"],
        "string": ["country", "state"]
    },
    "exchange_rates": {
        "date": ["date"],
        "string": ["currency"],
        "to_decimal": ["exchange"]
    }
}


In [ ]:
customers_bronze.filter(F.col('_rescued_data').isNotNull()).show(5, truncate=False)

In [ ]:
sales_bronze.show(5, truncate=False)

In [ ]:
sales_bronze.printSchema()

In [ ]:
# Checking there are values containing letters,so that I can turn them to int
sales_bronze.filter(F.expr("try_cast(order_number as int)").isNull()).show(truncate=False)

In [ ]:
products_bronze.show(5, truncate=False)

In [ ]:
products_bronze.printSchema()

In [ ]:
stores_bronze.show(5, truncate=False)

In [ ]:
stores_bronze.printSchema()

In [ ]:
exchange_rates_bronze.show(5, truncate=False)

In [ ]:
SILVER_TYPES = {
    "customers" : {
        "int": ["customerkey"],
        "date": ["birthday"],
        "string": ["gender", "name", "city", "state_code", "state", "zip_code", "country", "continent"]
    },
    "sales": {
        "int": ["order_number", "line_item", "customerkey", "storekey", "productkey", "quantity"],
        "date": ["order_date", "delivery_date"],
        "string": ["currency_code"]
    },
    "products": {
        "int": ["productkey", "subcategorykey", "categorykey"],
        "string": ["product_name", "brand", "color", "subcategory", "category"]
    },
    "stores": {
        "int": ["storekey", "square_meters"],
        "date": ["open_date"],
        "string": ["country", "state"]
    },
    "exchange_rates": {
        "date": ["date"],
        "string": ["currency"],
    }
}


def clean_money(col_name):
    return F.regexp_replace(F.col(col_name), r"[$,\s]", "").cast("decimal(10,2)")

def to_int(df, cols):
    for col in df.columns:
        if col in cols:
            df = df.withColumn(col, F.col(col).cast("int"))

    return df

def to_date(df, cols, fmt="M/d/yyyy"):

    for col in df.columns:
        if col in cols:
            before_count = df.filter(F.col(col).isNotNull()).count()
            df = df.withColumn(col, F.try_to_date(F.col(col), fmt))
            after_count = df.filter(F.col(col).isNotNull()).count()

            if after_count < before_count:
                raise ValueError(f"{col}: {before_count - after_count} values failed to parse as {fmt}")

    return df

def str_clean(df, cols):
    for col in df.columns:
        if col in cols:
            df = df.withColumn(col, F.trim(F.col(col)))

    return df

def apply_types(df, types_dict):
    d_type_mapper =  {
    "int": to_int,
    "date": to_date,
    "string": str_clean,
}
    for d_type, cols in types_dict.items():
        df = d_type_mapper[d_type](df, cols)

    return df


def type_sales(df):
    df = apply_types(df, SILVER_TYPES["sales"])

    return df

def type_customers(df):
    df = apply_types(df, SILVER_TYPES["customers"])

    return df.withColumn("age", F.floor(F.months_between(F.current_date(), "birthday") / 12))

def type_products(df):
    df = apply_types(df, SILVER_TYPES["products"])
    return (
        df
        .withColumn("unit_price_usd", clean_money("unit_price_usd"))
        .withColumn("unit_cost_usd", clean_money("unit_cost_usd"))
    )


def type_stores(df):
    df = apply_types(df, SILVER_TYPES["stores"])

    return df


def type_exchange_rates(df):
    df = apply_types(df, SILVER_TYPES["exchange_rates"])

    return (
        df
        .withColumn("exchange", F.col("exchange").cast("decimal(10,4)"))
    )

tables = ["sales", "customers", "products", "stores", "exchange_rates"]

clean_mapper = {
    "sales": type_sales,
    "customers": type_customers,
    "products": type_products,
    "stores": type_stores,
    "exchange_rates": type_exchange_rates,
}

cleaned_tables = {}

for table in tables:
    df = spark.table(f"electronics.bronze.{table}")
    df = clean_mapper[table](df)

    cleaned_tables[table] = df


for name, df in cleaned_tables.items():
    print(f"{name}:")
    df.printSchema()



In [ ]:
cleaned_tables["products"].show(5, truncate=False)

In [ ]:
cleaned_tables["exchange_rates"].show(5, truncate=False)

In [ ]:
cleaned_tables["stores"].show(5, truncate=False)

In [ ]:
cleaned_tables["sales"].show(5, truncate=False)

In [ ]:
cleaned_tables["customers"].show(5, truncate=False)

In [ ]:
cleaned_tables["products"].select("unit_price_usd", "unit_cost_usd").show(5, truncate=False)

In [ ]:
cleaned_tables["products"].filter(F.col("unit_price_usd").isNull()).count()

In [ ]:
cleaned_tables["products"].filter(F.col("unit_cost_usd").isNull()).count()

In [ ]:
cleaned_tables["exchange_rates"].filter(F.col("exchange").isNull()).count()

In [ ]:
from src.silver import build_silver

build_silver(spark, "electronics", "2026-09-18", "sales")

spark.table("electronics.silver.sales").printSchema()

In [ ]:
for entity in ["sales", "customers", "products", "stores", "exchange_rates"]:
    print(entity, build_silver(spark, "electronics", "2026-09-18", entity))

In [ ]:
cleaned_tables["customers"].show(5, truncate=False)

In [ ]:
sales_df = cleaned_tables["sales"]

date_range = (
    sales_df
    .agg(
        F.least(
            F.min(sales_df["order_date"]),
            F.min(sales_df["delivery_date"]),
        ).alias("min_date"),
        F.greatest(
            F.max(sales_df["order_date"]),
            F.max(sales_df["delivery_date"]),
        ).alias("max_date"),
    )
)

date_range.show()

In [ ]:
dim_date = (
    spark.sql("""
    SELECT explode(
            sequence(
            to_date('2015-01-01'),
            to_date('2026-12-31'),
            interval 1 day
            )
    ) AS full_date
    """)
    .select(
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_sk"),
        "full_date",
        F.dayofmonth("full_date").alias("day_of_month"),
        F.date_format("full_date", "EEEE").alias("day_name"),
        F.weekofyear("full_date").alias("week_of_year"),
        F.month("full_date").alias("month_number"),
        F.date_format("full_date", "MMMM").alias("month_name"),
        F.quarter("full_date").alias("quarter"),
        F.year("full_date").alias("year"),
        F.when(
            F.dayofweek("full_date").isin([1, 7]),
            True
        ).otherwise(False).alias("is_weekend"),
    )
)

dim_date.show(5, truncate=False)

In [ ]:
dim_date.filter(F.col("is_weekend")).select("full_date", "day_name").show(5)
dim_date.count()

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS electronics.gold")
dim_date.write.mode("overwrite").saveAsTable("electronics.gold.dim_date")
spark.table("electronics.gold.dim_date").count()

In [ ]:
dim_product = (
    spark.table("electronics.silver.products")
    .withColumn("product_sk", F.xxhash64(F.col("productkey").cast("string")))
    .drop("_rescued_data", "_file_path", "_file_size", "_last_modified_at", "_ingested_at", "_run_date", "_silver_run_date")
)


dim_product.count()

In [ ]:
dim_product.write.mode("overwrite").saveAsTable("electronics.gold.dim_product")

d = spark.table("electronics.gold.dim_product")

print(d.count())
print(d.select("product_sk").distinct().count())